# GPU Training with CuPy

NeuraCommand supports GPU acceleration via CuPy, a drop-in NumPy replacement for NVIDIA GPUs.

**Requirements:**
- NVIDIA GPU with CUDA support
- CuPy installed: `pip install cupy-cuda12x` (adjust for your CUDA version)

**Benefits:**
- 10-100x faster training on large datasets
- Same API as CPU training
- Easy device transfers

## 1. Check GPU Availability

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '../src')

from neura_command.backend import is_cupy_available, get_backend

HAS_GPU = is_cupy_available()

if HAS_GPU:
    import cupy as cp
    print("✓ CuPy is available!")
    print(f"  GPU: {cp.cuda.runtime.getDeviceCount()} device(s)")
    for i in range(cp.cuda.runtime.getDeviceCount()):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  Device {i}: {props['name'].decode()}")
else:
    print("✗ CuPy not available. Running in CPU-only mode.")
    print("  Install with: pip install cupy-cuda12x")
    print("  (Adjust cuda12x to match your CUDA version)")

## 2. Create Models for CPU and GPU

In [ ]:
from neura_command import Sequential, Dense, ReLU, Softmax, Adam, CrossEntropyLoss

def create_model(device='cpu'):
    """Create a model on the specified device."""
    return Sequential(
        layers=[
            Dense(784, 256, device=device),
            ReLU(device=device),
            Dense(256, 128, device=device),
            ReLU(device=device),
            Dense(128, 10, device=device),
            Softmax(device=device),
        ],
        optimizer=Adam(learning_rate=0.001),
        loss=CrossEntropyLoss(),
        device=device,
    )

# Create CPU model
model_cpu = create_model('cpu')
print("CPU Model:")
print(model_cpu.summary())

In [ ]:
# Create GPU model (if available)
if HAS_GPU:
    model_gpu = create_model('cuda')
    print("GPU Model:")
    print(model_gpu.summary())
else:
    print("Skipping GPU model (CuPy not available)")
    model_gpu = None

## 3. Generate Training Data

In [ ]:
# Generate synthetic MNIST-like data
np.random.seed(42)

n_samples = 10000
n_features = 784  # 28x28 images
n_classes = 10

# Random data (simulating images)
X = np.random.randn(n_samples, n_features).astype(np.float32)
y_labels = np.random.randint(0, n_classes, n_samples)
y = np.eye(n_classes)[y_labels].astype(np.float32)

print(f"Dataset: {n_samples} samples")
print(f"Features: {n_features}")
print(f"Classes: {n_classes}")

## 4. Training Speed Comparison

In [ ]:
import time

def benchmark_training(model, X, y, epochs=5, batch_size=64):
    """Benchmark training time."""
    start = time.perf_counter()
    model.train(X, y, epochs=epochs, batch_size=batch_size, verbose=False)
    elapsed = time.perf_counter() - start
    return elapsed

# Benchmark CPU
print("Training on CPU...")
model_cpu = create_model('cpu')  # Fresh model
time_cpu = benchmark_training(model_cpu, X, y, epochs=5)
print(f"CPU time: {time_cpu:.2f}s")

In [ ]:
# Benchmark GPU
if HAS_GPU:
    print("Training on GPU...")
    model_gpu = create_model('cuda')  # Fresh model
    time_gpu = benchmark_training(model_gpu, X, y, epochs=5)
    print(f"GPU time: {time_gpu:.2f}s")
    print(f"Speedup:  {time_cpu / time_gpu:.1f}x")
else:
    print("Skipping GPU benchmark (CuPy not available)")
    time_gpu = None

## 5. Device Transfer (CPU ↔ GPU)

In [ ]:
# Train on GPU, then transfer to CPU for deployment
if HAS_GPU:
    # Create and train on GPU
    model = create_model('cuda')
    model.train(X, y, epochs=10, batch_size=64, verbose=False)
    
    print(f"Trained on: {model.device}")
    
    # Transfer to CPU
    model.to('cpu')
    print(f"Now on: {model.device}")
    
    # Predict on CPU
    preds = model.predict(X[:10])
    print(f"Prediction shape: {preds.shape}")
else:
    print("Device transfer demo skipped (CuPy not available)")

## 6. Using the to() Method

In [ ]:
# Create model on CPU
model = create_model('cpu')
print(f"Initial device: {model.device}")

if HAS_GPU:
    # Move to GPU
    model.to('cuda')
    print(f"After to('cuda'): {model.device}")
    
    # Train on GPU
    model.train(X, y, epochs=3, batch_size=64, verbose=False)
    
    # Move back to CPU
    model.to('cpu')
    print(f"After to('cpu'): {model.device}")
else:
    print("to() demo skipped (CuPy not available)")

## 7. Automatic Data Transfer

In [ ]:
# When you pass NumPy arrays to a GPU model, they're auto-transferred
if HAS_GPU:
    model = create_model('cuda')
    
    # X and y are NumPy arrays (CPU)
    print(f"Input type: {type(X).__module__}")
    
    # Model automatically transfers data to GPU
    model.train(X, y, epochs=3, batch_size=64, verbose=False)
    
    # Predict returns NumPy array (CPU)
    preds = model.predict(X[:5])
    print(f"Output type: {type(preds).__module__}")
    print("Data transfer is automatic!")
else:
    print("Automatic transfer demo skipped (CuPy not available)")

## 8. Save/Load with Device

In [ ]:
import tempfile
import os

# Models are always saved to CPU-compatible format
if HAS_GPU:
    model = create_model('cuda')
    model.train(X, y, epochs=5, batch_size=64, verbose=False)
    
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, 'model')
        
        # Save (weights converted to NumPy)
        model.save(path)
        print(f"Saved GPU-trained model")
        
        # Load on CPU
        loaded = Sequential.load(path)
        print(f"Loaded on: {loaded.device}")
        
        # Move to GPU if needed
        loaded.to('cuda')
        print(f"Moved to: {loaded.device}")
else:
    # CPU-only example
    model = create_model('cpu')
    model.train(X, y, epochs=5, batch_size=64, verbose=False)
    
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, 'model')
        model.save(path)
        loaded = Sequential.load(path)
        print(f"Saved and loaded on: {loaded.device}")

## 9. GPU Memory Tips

In [ ]:
if HAS_GPU:
    print("GPU Memory Tips:")
    print()
    print("1. Use smaller batch sizes if you get OOM errors")
    print("   model.train(X, y, batch_size=32)  # instead of 256")
    print()
    print("2. Move model to CPU when not training")
    print("   model.to('cpu')")
    print()
    print("3. Clear GPU memory cache")
    print("   import cupy; cupy.get_default_memory_pool().free_all_blocks()")
    print()
    
    # Show current GPU memory usage
    mempool = cp.get_default_memory_pool()
    print(f"Current GPU memory: {mempool.used_bytes() / 1024 / 1024:.1f} MB")
    print(f"Total GPU memory:   {mempool.total_bytes() / 1024 / 1024:.1f} MB")
else:
    print("GPU memory tips (for when you have a GPU):")
    print("- Use smaller batch sizes for large models")
    print("- Move models to CPU when not training")
    print("- Clear GPU cache with cupy.get_default_memory_pool().free_all_blocks()")

## 10. Backend Functions Reference

In [ ]:
from neura_command.backend import (
    get_backend, 
    to_device, 
    to_numpy, 
    is_cupy_available,
)

print("Backend API:")
print()
print("is_cupy_available()")
print(f"  -> {is_cupy_available()}")
print()
print("get_backend('cpu')")
print(f"  -> {get_backend('cpu')}")
print()

if HAS_GPU:
    print("get_backend('cuda')")
    print(f"  -> {get_backend('cuda')}")
    print()
    
    # Transfer examples
    x_cpu = np.array([1.0, 2.0, 3.0])
    x_gpu = to_device(x_cpu, 'cuda')
    x_back = to_numpy(x_gpu)
    
    print("to_device(x, 'cuda')")
    print(f"  -> {type(x_gpu).__module__}.{type(x_gpu).__name__}")
    print()
    print("to_numpy(x_gpu)")
    print(f"  -> {type(x_back).__module__}.{type(x_back).__name__}")

## Summary

### GPU Training Workflow

```python
from neura_command import Sequential, Dense, ReLU, Softmax, Adam
from neura_command.backend import is_cupy_available

# Choose device
device = 'cuda' if is_cupy_available() else 'cpu'

# Create model on device
model = Sequential(
    layers=[...],
    optimizer=Adam(),
    device=device,
)

# Train (data auto-transferred)
model.train(X, y, epochs=100)

# Move to CPU for deployment
model.to('cpu')

# Save
model.save('model')
```

### Key Points

1. **Install CuPy**: `pip install cupy-cuda12x`
2. **Check availability**: `is_cupy_available()`
3. **Create on GPU**: `device='cuda'`
4. **Transfer devices**: `model.to('cuda')` or `model.to('cpu')`
5. **Auto-transfer**: NumPy data is auto-transferred to GPU
6. **Predictions**: Always return NumPy arrays (CPU)